In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import random

In [2]:
class Head(nn.Module):

    def __init__(self, x_emb, head_emb):
        super().__init__()
        self.k = nn.Linear(x_emb, head_emb)
        self.q = nn.Linear(x_emb, head_emb)
        self.v = nn.Linear(x_emb, head_emb)

    def forward(self, x):
        k = self.k(x)                                   # (seq_length, head_emb)
        q = self.q(x)                                   # (seq_length, head_emb)
        v = self.v(x)                                   # (seq_length, head_emb)

        x = q @ k.transpose(-2, -1)                     # (seq_length, seq_length) we have not used q@k.T since it will be invalid operation in case of batches
        x = x/ pow(k.shape[-1], 0.5)
        x = F.softmax(x, dim=-1)                        # (seq_length, seq_length)
        x = x @ v                                       # (seq_length, head_emb)
        return x


In [8]:
class MultiHeadAttention(nn.Module):

    def __init__(self, x_emb, heads_num, head_emb):
        super().__init__()
        self.heads_num = heads_num
        self.heads = nn.ModuleList([Head(x_emb, head_emb) for _ in range(heads_num)])
        self.proj = nn.Linear(x_emb, x_emb)

    def forward(self, tokens):
        x = torch.cat([head.forward(tokens) for head in self.heads], dim=-1)
        x = self.proj(x)
        return x
        

In [9]:
class MultiheadBlock(nn.Module):

    def __init__ (self, heads_num, seq_length, x_emb):
        super().__init__()
        self.layer_norm = nn.LayerNorm((seq_length, x_emb))
        self.heads = MultiHeadAttention(x_emb, heads_num, x_emb//heads_num)

    def forward(self, tokens):
        x = self.heads(tokens)
        x = self.layer_norm(x)
        return tokens + x

In [87]:
class FeedFwdBlock(nn.Module):

    def __init__(self, seq_length, x_emb):
        super().__init__()
        self.layer = nn.Linear(x_emb, x_emb)
        self.layer_norm = nn.LayerNorm((seq_length, x_emb))

    def forward(self, tokens):
        x = self.layer(tokens)
        x = self.layer_norm(x)
        return tokens + x

In [ ]:
class Encoder(nn.Module):

    def __init__(self, vocab_size: int, x_emb: int, seq_len: int, heads_num: int, special_tokens = {}):
        super().__init__()
        self.seq_len = seq_len
        self.special_tokens = special_tokens
        self.look_up_table = torch.randn((vocab_size+2, x_emb))
        self.postional_enc = torch.randn((x_emb, seq_len))
        self.optimizer = torch.optim.AdamW(self.parameters(), lr=1e-3)
        self.architecture = nn.ModuleList([
            MultiheadBlock(heads_num, seq_len, x_emb),
            FeedFwdBlock(seq_len, x_emb)
        ])


    def forward(self, tokens, targets = None):
        # implement tokenizer here
        tokens = torch.tensor(tokens)

        loss = None

        # lookup_table + postional_enc
        x = self.look_up_table[tokens] + (tokens * self.postional_enc).T

        for block in self.architecture:
            x = block(x)
        
        if targets:
            B, T, C = x.shape
            x = x.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(x, targets)
        
        return x, loss
    
    def training(self, tokens, epochs = 100, batch_size = 32):
        chunks = []
        for i in range(0, len(tokens) - self.seq_len, self.seq_len):
            chunks.append(tokens[i : i + self.seq_len])

        last = tokens[len(chunks) * self.seq_len:]
        if len(last) > 0:
            last = last + [0] * (self.seq_len - len(last))
            chunks.append(last)

        x_chunks = [chunk[:-1] for chunk in chunks]
        y_chunks = [chunk[1:]  for chunk in chunks]


        for epoch in range(epochs):
            combined = list(zip(x_chunks, y_chunks))
            random.shuffle(combined)
            x_shuffled, y_shuffled = zip(*combined)
            x, y = list(x_shuffled), list(y_shuffled)

            x_batch = []
            y_batch = []

            i = random.randint(0, len(x) - batch_size)
            x_batch = x[i : i + batch_size]
            y_batch = y[i : i + batch_size]


            output, loss = self.forward(x_batch, y_batch)
            loss.backward()
            self.optimizer.step()
            self.optimizer.zero_grad()




In [91]:
test_encoder = Encoder(300, 20, 100, 4, {"<|endoftext|>": 50256, "<|padding|>": 50257})
test_encoder.forward("Hi bharat")

(tensor([[  88.1015,   47.9351,  -80.7808,  ...,  -79.0098,   90.6976,
            60.2874],
         [ 235.8005,   51.3472,  -70.0679,  ..., -109.7427,   61.5908,
          -134.1626],
         [ -32.6427,    2.9807,   -8.9561,  ...,   16.2438,  -12.8977,
            14.5974],
         ...,
         [ 453.2976, -281.6981,  215.8302,  ...,  110.0209, -277.9093,
          -322.1542],
         [ 303.6456,  -79.4413, -210.2665,  ...,  -37.9673,  233.0203,
           378.6934],
         [  -7.2692,  -65.6692,  208.5428,  ...,  -37.1369,  612.9870,
           286.3049]], grad_fn=<AddBackward0>),
 None)